# 01 • เรียกใช้ LLM API

เป้าหมาย: ทดลอง Prompt → Model → Output ด้วยรายงานรับน้ำยางและส่งมอบงาน แล้วอ่านจำนวน tokens

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ SOP ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

บริบทงานอ้างอิงจาก [เว็บไซต์กลุ่มศรีตรัง](https://www.sritranggroup.com/th/home) ส่วนรูปแบบ notebook อ้างอิงจาก [ตัวอย่าง GitHub](https://github.com/biodatlab/cpf-genai-workshop/tree/main/tutorial_notebooks) และใช้ [OpenAI API documentation](https://developers.openai.com/api/docs) สำหรับ syntax

## เตรียมพร้อมก่อนเริ่ม
เพิ่ม Secret ชื่อ `OPENAI_API_KEY` ใน Colab ด้วยตนเอง และเปิด **Notebook access** สำหรับ notebook นี้ โค้ดจะอ่านผ่าน `userdata.get()` โดยไม่แสดงค่า key

รันเซลล์จากบนลงล่าง หลังติดตั้ง package และเตรียม key แล้ว ทดสอบตัวอย่างหลักใน Colab ด้วย API แล้ว แต่ผล LLM อาจเปลี่ยนได้ การผ่านตัวอย่างนี้ไม่ใช่การรับรองทุกคำถามหรือการใช้งานจริง

ใช้ `gpt-4.1-mini` สำหรับสร้างคำตอบ (ไม่เกิน 500 tokens ต่อ request) และ `text-embedding-3-small` สำหรับ Embedding โดย Agent Loop จำกัดไว้ 5 รอบ

In [6]:
%pip -q install openai

In [7]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "gpt-4.1-mini"

## ส่ง Prompt แรก
`instructions` กำหนดบทบาทและรูปแบบคำตอบ ส่วน `input` คือข้อมูลหรืองานที่ต้องการให้ทำ

In [9]:
prompt = """ช่วยสรุปบันทึกส่งมอบงานกะเช้า:
รับน้ำยาง Lot LTX-001 จากผู้ขายสมมติ 8,000 กก. รอผล QC
คลังยางแท่งเตรียม STR20 จำนวน 2 Lot สำหรับส่งมอบ
รถรับสินค้าล่าช้า 1 ชั่วโมง ฝ่ายขนส่งกำลังยืนยันเวลาใหม่"""
response = client.responses.create(
    model=MODEL,
    instructions="คุณเป็นผู้ช่วยประสานงาน สรุปเป็นภาษาไทยเป็น bullet points เพียง 3 ข้อ: 1. รับน้ำยางและสถานะ QC 2. เตรียมยางแท่ง 3. รถรับสินค้า ในแต่ละข้อระบุงานที่เสร็จแล้วและงานที่ต้องติดตามเท่าที่ข้อความให้ ห้ามเติมข้อเท็จจริง",
    input=prompt,
    max_output_tokens=500,
)
print(response.output_text)
print(response.usage)

- รับน้ำยางและสถานะ QC: รับน้ำยาง Lot LTX-001 จำนวน 8,000 กก. จากผู้ขายสมมติ เสร็จแล้ว แต่ยังรอผลตรวจสอบคุณภาพ (QC)  
- เตรียมยางแท่ง: คลังยางแท่งเตรียมยาง STR20 จำนวน 2 Lot สำหรับส่งมอบ เสร็จแล้ว  
- รถรับสินค้า: รถรับสินค้าล่าช้า 1 ชั่วโมง งานที่ต้องติดตามคือฝ่ายขนส่งกำลังยืนยันเวลาใหม่
ResponseUsage(input_tokens=179, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=120, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=299)


## ลองทำด้วยตนเอง
1. เปลี่ยนจากสรุปรายงานเป็นจัดหมวดหมู่ประเด็น: รับซื้อ / QC / ขนส่ง
2. ขอคำตอบเป็น 3 bullet points พร้อมผู้รับผิดชอบ
3. เปลี่ยนเป็นร่างข้อความส่งต่องานให้กะถัดไป แล้วเปรียบเทียบ tokens